### The Production Lifecycle:
1. **Harvesting**: Generating/Extracting raw content (simulating 30 docs).
2. **Deep Ingestion**: Propositional Decomposition (The "Atomic" Facts).
3. **PDR Indexing**: Linking precision child vectors to rich parent context.


## Phase 0: Data Harvesting (Preparing our 30 Documents)
In a real project, your data lives in folders or buckets. We will use Python to generate a **Complex Knowledge Base** of 30 product manuals and support logs.


In [ ]:
import os

# 1. Create the persistent knowledge folder
kb_path = "data/knowledge_base"
os.makedirs(kb_path, exist_ok=True)

# 2. Generate 30 Realistic Tech Documents
# We define distinct 'Product Personas' to create a realistic search space
docs = [
    # --- LAPTOPS (High Search Overlap) ---
    {"name": "Aether-X1-Flagship", "content": "Flagship laptop with liquid-cooled RTX 4090 and 16-inch OLED. Designed for 8K video production. SN: X1-001."},
    {"name": "Aether-X1-Air", "content": "Travel-optimized X1 variant. Air-cooled RTX 4070. Magnesium chassis for weight reduction. SN: X1-AIR."},
    {"name": "Aether-Carbon-Pro", "content": "Workstation grade laptop. 128GB RAM. Features dual thermal management loops. Color: Midnight Grey."},
    # --- SUPPORT LOGS (Semantic Noise) ---
    {"name": "Support-Log-GPU", "content": "Ticket #404: User reports GPU thermal throttling on X1-Flagship. Recommended solution: Flush liquid reservoir."},
    {"name": "FAQ-Battery", "content": "How to calibrate battery? Discharge to 5% and then charge to 100% using the 240W GaN charger."},
    # --- TABLETS ---
    {"name": "Nebula-Tab-12", "content": "Nebula Tablet 12-inch. 120Hz Refresh Rate. Sub-10ms latency for digital stylus users. Target: Artists."},
    {"name": "Nebula-Tab-Lite", "content": "Budget tablet for education. 60Hz LCD. 10-hour battery. No stylus support. SN: NT-LITE."}
]

# Bulk up to 30 with variations
for i in range(len(docs), 31):
    docs.append({"name": f"Product-Bulletin-{i}", "content": f"Bulletin {i}: Technical update for peripheral model {i*10}. Firmware version {i}.2 released."})

for doc in docs:
    with open(f"{kb_path}/{doc['name']}.txt", "w") as f:
        f.write(doc['content'])

print(f"✅ Generated {len(os.listdir(kb_path))} high-fidelity documents in {kb_path}")


## Phase 1: Deep Ingestion (Crawling the Filesystem)
Now, we crawl our 30 documents and decompose them into **Atomic Facts**.


In [ ]:
import numpy as np
from redisvl.index import SearchIndex
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

class DeepIngestionEngine:
    '''Simulates a production pipeline that extraction atomic facts.'''
    @staticmethod
    def extract_propositions(text):
        return [s.strip() for s in text.split(". ") if len(s) > 10]

# --- THE INDEXING ---
schema = {
    "index": {"name": "pdr_idx", "prefix": "chunk:"},
    "fields": [
        {"name": "parent_id", "type": "tag"},
        {"name": "proposition", "type": "text"},
        {"name": "embedding", "type": "vector", "attrs": {"dims": 384, "algorithm": "flat"}}
    ]
}
index = SearchIndex.from_dict(schema, redis_url="redis://redis-vector-db:6379")
index.create(overwrite=True)

# --- THE CRAWLER ---
docs_to_load = []
raw_knowledge = {} # For PDR retrieval storage

for filename in os.listdir(kb_path):
    with open(f"{kb_path}/{filename}", "r") as f:
        text = f.read()
        doc_id = filename.replace(".txt", "")
        raw_knowledge[doc_id] = text # Store the 'Parent'
        
        facts = DeepIngestionEngine.extract_propositions(text)
        for fact in facts:
            docs_to_load.append({
                "parent_id": doc_id,
                "proposition": fact,
                "embedding": embedder.encode(fact).tolist()
            })

index.load(docs_to_load)
print(f"✅ Bulk Ingestion Complete: {len(docs_to_load)} facts indexed from {len(os.listdir(kb_path))} raw files.")


## The PDR Retrieval Loop
We find the fact, but we fetch the source.


In [ ]:
from redisvl.query import VectorQuery

def pdr_search(query):
    q_vec = embedder.encode(query).tolist()
    vq = VectorQuery(q_vec, "embedding", return_fields=["parent_id", "proposition"], num_results=1)
    results = index.query(vq)
    
    if results:
        parent_id = results[0]['parent_id']
        # Fetch the FULL context from our high-fidelity storage (Representation)
        full_context = raw_knowledge[parent_id]
        return full_context, results[0]['proposition']
    return None, None

context, matched_fact = pdr_search("How is the GPU in the Aether cooled?")
print(f"Matched Atomic Fact: {matched_fact}")
print(f"Full Context Returned: {context}")


# Day 2 | Notebook 5b: Multi-Modal Search
**Author: Sattaya Singkul**

Searching with text is Day 1. Searching with **Images + Text** and merging their meanings is Day 2.

### The Joint Space Pattern
We will implement a **Weighted Multi-Vector Ranker**. This searches two different fields and merges scores based on a priority (e.g., Image matches are 70% of the score).


In [ ]:
import numpy as np

class MultiModalSearchService:
    def __init__(self, text_weight=0.3, image_weight=0.7):
        self.tw = text_weight
        self.iw = image_weight
        
    def perform_search(self, text_query_vec, image_query_vec):
        # Simulation: A diversified asset library
        results = [
            {"name": "Aether X1 (Black)", "text_sim": 0.85, "img_sim": 0.90},
            {"name": "Aether X1 (Silver)", "text_sim": 0.82, "img_sim": 0.45},
            {"name": "Nebula Tablet (Grey)", "text_sim": 0.40, "img_sim": 0.88},
            {"name": "Office Chair (Ergo)", "text_sim": 0.10, "img_sim": 0.15},
            {"name": "Pulse Watch (Red)", "text_sim": 0.30, "img_sim": 0.95}
        ]
        
        final_scores = []
        for r in results:
            merged_score = (r['text_sim'] * self.tw) + (r['img_sim'] * self.iw)
            final_scores.append({"item": r['name'], "score": merged_score})
            
        return sorted(final_scores, key=lambda x: x['score'], reverse=True)

# --- EXECUTION ---
searcher = MultiModalSearchService()
# Imagine these are CLIP embeddings for a "Black Laptop"
ranking = searcher.perform_search([0.1]*512, [0.9]*512)
print("Weighted Multi-Modal Ranking (Prioritizing Images):")
for rank, match in enumerate(ranking[:3]):
    print(f"Rank {rank+1}: {match['item']} | Combined Score: {match['score']:.4f}")


# Day 2 | Notebook 5c: RAG Evaluation & Observability
**Author: Sattaya Singkul**

Build it. Measure it. Guard it.

### Core Objectives:
1. **Faithfulness**: Does the answer come ONLY from the context?
2. **Relevancy**: Does the context actually answer the user's question?
3. **The Hallucination Guard**: A production wrapper that blocks "unsafe" answers.


In [ ]:
class RAGAnalyzer:
    '''Calculates Faithfulness and Relevancy for RAG turns.'''
    @staticmethod
    def check_faithfulness(answer, context):
        # (Simplified implementation)
        # Check if key tokens in the answer exist in the context
        context_words = set(context.lower().split())
        answer_words = [w for w in answer.lower().split() if len(w) > 3]
        hits = sum(1 for w in answer_words if w in context_words)
        return hits / len(answer_words) if answer_words else 0.0

    @staticmethod
    def check_relevancy(query, context):
        # (In production, use Cross-Encoder here)
        # Higher score means the context is highly pertinent to the question.
        return 0.95 # Mock high relevancy result

# --- PRODUCTION GUARD ---
def hallucination_guard(query, context, answer, threshold=0.6):
    analyzer = RAGAnalyzer()
    
    f_score = analyzer.check_faithfulness(answer, context)
    r_score = analyzer.check_relevancy(query, context)
    
    print(f"Audit Result -> Faithfulness: {f_score:.2f} | Relevancy: {r_score:.2f}")
    
    if f_score < threshold:
        return "🚨 REFUSAL: I found the information, but I cannot verify it with 100% precision. Please refer to support."
    return answer

# --- THE TEST ---
mock_q = "Specs of Aether laptop?"
mock_c = "The Aether laptop has 64GB of RAM and an OLED screen."
mock_a = "It has 64GB of RAM and runs on solar power." # Solar power is a hallucination!

guarded_answer = hallucination_guard(mock_q, mock_c, mock_a)
print(f"\nFinal Agent Output: {guarded_answer}")
